# Sarvam Models Cookbook - 30B & 105B

**Provider:** Sarvam AI  |  **Models:** `sarvam-30b` · `sarvam-30b-16k` · `sarvam-105b`  |  **License:** Apache 2.0  
**Docs (30B):** https://docs.sarvam.ai/api-reference-docs/getting-started/models/sarvam-30b  
**Docs (105B):** https://docs.sarvam.ai/api-reference-docs/getting-started/models/sarvam-105b

---

## About Sarvam-30B

Sarvam-30B is a 30B-parameter Mixture-of-Experts (MoE) chat LLM trained from scratch with only **2.4B active parameters per token**. Delivers state-of-the-art Indian language performance alongside strong reasoning, coding, and conversational capabilities.

| Feature | Details |
|---|---|
| **Architecture** | MoE - 128 sparse experts; 2.4B active params/token; 3×–6× throughput vs dense on H100 |
| **Context window** | 64K tokens (`sarvam-30b`) · 16K tokens (`sarvam-30b-16k`) |
| **Indian Languages** | 10 languages - native script, romanized, and code-mixed (Hinglish, etc.) |
| **Hybrid Thinking Mode** | `reasoning_effort` = `low` / `medium` / `high` - step-by-step chain-of-thought |
| **Wiki Grounding** | `wiki_grounding=True` anchors responses to Wikipedia-sourced facts |
| **Tool Calling** | JSON schema function definitions; structured `tool_calls` in response |
| **API compatibility** | OpenAI-compatible - drop-in via base-URL override |
| **Math500** | 97.0 |
| **AIME 25** | 88.3 |
| **Indian Language Win Rate** | 89% pairwise comparisons |

---

## About Sarvam-105B

Sarvam-105B is Sarvam's **flagship** Mixture-of-Experts model with Multi-head Latent Attention (MLA). Use it for highest-quality outputs, complex multi-step reasoning, or production agentic workflows.

| Feature | Details |
|---|---|
| **Architecture** | MoE + Multi-head Latent Attention (MLA) - 128 sparse experts |
| **Total parameters** | 105B+ |
| **Pre-training data** | 12T tokens - code, math, multilingual, web |
| **Hybrid Thinking Mode** | `reasoning_effort` = `low` / `medium` / `high` - same parameter as 30B |
| **Wiki Grounding** | `wiki_grounding=True` - supported, same as 30B |
| **API compatibility** | Fully drop-in - same API surface as 30B, just change the model ID |
| **Math500** | 98.6 |
| **AIME 25** | 88.3 (96.7 with tools) |
| **BrowseComp** | 49.5 (vs 35.5 for 30B) |
| **Indian Language Win Rate** | 90% pairwise comparisons |
| **Inference** | Server-centric - H100 optimized (not edge-deployable) |

---

## Choosing Between Models

| | Sarvam-30B / 30B-16k | Sarvam-105B |
|---|---|---|
| **Best for** | Real-time, conversational, cost-sensitive | Complex reasoning, agentic, highest quality |
| **Parameters** | 30B (2.4B active) | 105B+ |
| **Context** | 64K / 16K tokens | Server-default(128k) |
| **Math500** | 97.0 | 98.6 |
| **BrowseComp** | 35.5 | 49.5 |
| **Indian Language Win Rate** | 89% | 90% |
| **Inference targets** | H100, L40S, Apple Silicon | H100 (server-centric) |

> All code in this cookbook works with **both models** - just swap `MODEL = 'sarvam-105b'` in the setup cell.

---

## Table of Contents

- **Section 0** - Setup & Installation
- **Section 1** - Basic Usage with Sarvam AI SDK
  - 1.1 Chat Completions API
  - 1.3 Multi-turn Conversation
  - 1.4 Streaming Responses
- **Section 2** - Framework Integration (LangChain)
  - 2.1 Basic Chat
  - 2.2 Prompt Templates & Chains
  - 2.3 Structured Output
  - 2.4 Tool Calling
- **Section 3** - Building Agents
  - 3.1 Basic Agent with 2 Tools
  - 3.2 Multi-Agent Handoffs
  - 3.3 Agents as Tools
  - 3.4 Agent with Input Guardrails
  - 3.5 Wiki-Grounded Search Agent
  - 3.6 LangGraph Agent
  - 3.7 Native Tool Calling (Sarvam SDK)
- **Section 4** - Advanced Applications
  - 4.1 Structured Output via SDK + Pydantic
  - 4.2 Function Calling (Raw JSON Schema)
  - 4.3 RAG Pipeline
  - 4.4 Content Generation Pipeline
  - 4.5 Async Batch Processing
  - 4.6 JSON Mode
  - 4.7 Hybrid Thinking Mode (`reasoning_effort`) ★ Sarvam-unique
  - 4.8 Wiki Grounding ★ Sarvam-unique
  - 4.9 Indian Language Support ★ Sarvam-unique
- **★** Quick-Reference Footer


---
## Section 0 - Setup & Installation

Install the native Sarvam AI SDK, the OpenAI SDK (for streaming and OpenAI-compat calls), LangChain, LangGraph, and Pydantic.


In [ ]:
!pip install -Uqq sarvamai openai langchain langchain-openai langgraph pydantic

In [ ]:
import os
import re
import json
import asyncio
import warnings
from sarvamai import SarvamAI
from openai import OpenAI

warnings.filterwarnings('ignore')

# ── API key ────────────────────────────────────────────────────────────────────
# Set via environment variable or replace the placeholder below.
os.environ['SARVAM_API_KEY'] = 'sk-...'   # ← replace with your key

# ── Model selection ────────────────────────────────────────────────────────────
MODEL = 'sarvam-30b-16k'        # 30B - 16K-context variant (cost-efficient, real-time)
# MODEL = 'sarvam-30b'          # 30B - full 64K-context variant
# MODEL = 'sarvam-105b'         # 105B - flagship; best for complex reasoning & agentic workflows
#                               #        (MoE + MLA, 105B+ params, 98.6 Math500, 49.5 BrowseComp)

SARVAM_BASE_URL = 'https://api.sarvam.ai/v1'

# Native Sarvam AI SDK client
client = SarvamAI(api_subscription_key=os.environ['SARVAM_API_KEY'], timeout=300)

# OpenAI-compatible client (for streaming, JSON mode, async, and raw tool calls)
oa_client = OpenAI(
    api_key=os.environ['SARVAM_API_KEY'],
    base_url=SARVAM_BASE_URL,
    timeout=300,
)

print(f'Model configured: {MODEL}')


Model configured: sarvam-30b-16k


---
## Section 1 - Basic Usage with Sarvam AI SDK

All calls in this section use the native `sarvamai` Python package.

> **Note:** Section 1.2 (Responses API) is skipped - Sarvam AI does not expose a separate Responses API; `client.chat.completions()` is the unified entry point.


### 1.1 Chat Completions API

A single-turn request using `client.chat.completions()`. The response follows the OpenAI-compatible `choices[0].message.content` structure.


In [14]:
response = client.chat.completions(
    model=MODEL,
    messages=[
        {'role': 'user', 'content': 'Explain the significance of the Indian monsoon season in two sentences.'}
    ],
    temperature=0.5,
    top_p=1,
    max_tokens=2000,
)

print(response.choices[0].message.content)
print(f'\n--- Tokens used: {response.usage.total_tokens} ---')


The Indian monsoon is the lifeline of the nation's agriculture, supplying the rainfall necessary for its staple crops and rural livelihoods. Beyond farming, it replenishes rivers and groundwater, underpinning water security, power generation, and the country's overall economic stability.

[Reasoning used 5349 chars internally]

--- Tokens used: 1313 ---


### 1.3 Multi-turn Conversation

Build conversation history by appending assistant replies back into the `messages` list. The `system` role sets persistent behaviour across all turns.


In [15]:
conversation = [
    {'role': 'system', 'content': 'You are a knowledgeable guide on Indian culture and history.'},
    {'role': 'user',   'content': 'What is the Mahabharata?'},
]

# Turn 1
response1 = client.chat.completions(model=MODEL, messages=conversation, max_tokens=2000)
reply1 = response1.choices[0].message.content
conversation.append({'role': 'assistant', 'content': reply1})
print(f'Turn 1 - Assistant:\n{reply1}\n')

# Turn 2 - model has full context of the previous exchange
conversation.append({'role': 'user', 'content': 'Who are the two main warring factions in it?'})
response2 = client.chat.completions(model=MODEL, messages=conversation, max_tokens=2000)
print(f'Turn 2 - Assistant:\n{response2.choices[0].message.content}')


Turn 1 - Assistant:
Of course. The Mahabharata is one of the two major Sanskrit epics of ancient India, and it is arguably the longest poem ever written. It's far more than just a story; it's a vast encyclopedia of ancient Indian thought, culture, philosophy, and ethics.

Here’s a breakdown of what it is, from the surface story to its profound depths.

### 1. The Core Narrative: A Story of War and Family

At its heart, the Mahabharata is the story of a dynastic struggle for the throne of Hastinapura.

*   **The Main Players:** The central conflict is between two groups of cousins:
    *   **The Pandavas:** The five heroic and virtuous brothers-Yudhishthira, Bhima, Arjuna, Nakula, and Sahadeva-who are the rightful heirs to the throne.
    *   **The Kauravas:** The one hundred sons of the blind king Dhritarashtra, led by the eldest, Duryodhana, who are jealous of the Pandavas' prowess and popularity.

*   **The Plot:** The epic begins with the Pandavas' birth, their education, and their 

### 1.4 Streaming Responses

Streaming returns tokens as server-sent events via the OpenAI-compatible endpoint. Each chunk carries a `delta.content` fragment; iterate to reconstruct the full response in real time.


In [16]:
stream = oa_client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'List 3 classical Indian dance forms with a one-line description each.'}],
    stream=True,
    max_tokens=2000,
)

print('Streaming response:\n')
full = ''
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end='', flush=True)
        full += delta.content
print()  # final newline
if not full:
    print('[No streamed content - model may have output reasoning_content only]')


Streaming response:


1.  **Bharatanatyam:** A classical dance form from Tamil Nadu known for its precise movements, expressive gestures, and rhythmic footwork.
2.  **Kathak:** A North Indian classical dance form that emphasizes rhythmic footwork, storytelling, and expressive gestures, often drawing from Hindu mythology.
3.  **Kathakali:** A highly stylized and dramatic dance-drama from Kerala, recognized by its elaborate costumes, intricate makeup, and powerful, exaggerated gestures.


---
## Section 2 - Framework Integration (LangChain)

Sarvam-30B is OpenAI-compatible, so `ChatOpenAI` from `langchain-openai` works by overriding `openai_api_base`.


### 2.1 Basic Chat

Initialise `ChatOpenAI` with `openai_api_base` pointing to Sarvam's endpoint. This object is reused throughout Sections 2 and 3.


In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

lc_llm = ChatOpenAI(
    model=MODEL,
    openai_api_key=os.environ['SARVAM_API_KEY'],
    openai_api_base=SARVAM_BASE_URL,
    temperature=0.5,
)

messages = [
    SystemMessage(content='You are a helpful assistant specialising in Indian cuisine.'),
    HumanMessage(content='What are the key spices that define a classic biryani?'),
]
response = lc_llm.invoke(messages)
print(response.content)


Of course! A classic biryani is a symphony of spices, where each one plays a crucial role in creating its signature aroma, flavour, and colour. While the exact blend can vary by region and family, there are a core set of spices that are almost always present.

Here are the key spices that define a classic biryani, broken down by their function:

---

### The Foundation: The Whole Spices (The "Tadka")

These are typically bloomed in hot ghee or oil at the beginning of the cooking process to release their essential oils.

1.  **Cinnamon (Dalchini):** Adds a warm, sweet, and woody depth. It's a fundamental backbone of the biryani masala.
2.  **Cloves (Laung):** Provide a strong, pungent, and slightly sweet flavour. They are potent, so they are used in moderation.
3.  **Green Cardamom (Elaichi):** The most aromatic of the cardamoms. It lends a sweet, intense fragrance that is unmistakable in biryani.
4.  **Bay Leaf (Tej Patta):** Contributes a subtle, herbal, and slightly floral note that 

### 2.2 Prompt Templates & Chains

Use `ChatPromptTemplate` piped to the LLM and a `StrOutputParser` to build a reusable, parameterised chain.


In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an expert on Indian classical music.'),
    ('human', 'Explain {topic} at a {detail_level} level of detail.'),
])

chain = prompt | lc_llm | StrOutputParser()

result = chain.invoke({'topic': 'Raga Bhairav', 'detail_level': 'beginner'})
print(result)


Of course! Let's explore Raga Bhairav in a way that's easy to grasp, even if you've never heard of it before.

### Raga Bhairav: The King of Morning Ragas

Imagine the very first light of dawn breaking over the Ganges in Varanasi. The air is cool, quiet, and filled with a sense of immense peace and devotion. This is the world of Raga Bhairav.

In simple terms, **Raga Bhairav is a specific musical framework that is designed to evoke the feeling and mood of the early morning.** It's considered the "King of Morning Ragas" because it sets a majestic, serious, and deeply spiritual tone.

---

### The "Personality" of Bhairav

Every raga has a unique personality, much like a person. Bhairav's personality is:

*   **Solemn and Majestic:** It's not a light, happy raga. It has a weight and grandeur to it.
*   **Devotional:** Bhairav is deeply connected to Lord Shiva and is often used in prayers and meditative songs.
*   **Contemplative:** It encourages a calm, introspective state of mind, perfe

### 2.3 Structured Output

`with_structured_output()` instructs the model to return data conforming to a Pydantic `BaseModel`. The result is a typed Python object, not a raw string.


In [20]:
from pydantic import BaseModel, Field
from typing import List

class CulinaryDish(BaseModel):
    name: str = Field(description='Name of the dish')
    region: str = Field(description='Indian region where the dish originates')
    main_ingredients: List[str] = Field(description='Primary ingredients list')
    spice_level: str = Field(description='Spice level: mild, medium, or hot')

# Use json_mode for compatibility with the sarvam model's reasoning output
structured_llm = lc_llm.with_structured_output(CulinaryDish, method='json_mode')
dish = structured_llm.invoke(
    'Tell me about Dal Makhani. '
    'Respond ONLY with a JSON object matching this schema: '
    '{"name": str, "region": str, "main_ingredients": [str, ...], "spice_level": str}'
)

print(f'Dish:        {dish.name}')
print(f'Region:      {dish.region}')
print(f'Ingredients: {", ".join(dish.main_ingredients)}')
print(f'Spice level: {dish.spice_level}')


Dish:        Dal Makhani
Region:      Punjab
Ingredients: Urad Dal, Rajma, Butter, Cream, Tomatoes, Ginger-Garlic Paste, Cardamom, Cloves, Cinnamon, Bay Leaves, Garam Masala
Spice level: Mild


### 2.4 Tool Calling

Decorate Python functions with `@tool`, bind them to the LLM with `.bind_tools()`, and inspect `response.tool_calls` to see which functions the model chose to invoke.


In [21]:
from langchain_core.tools import tool

@tool
def get_festival_date(festival: str, year: int) -> str:
    """Return the approximate date of a major Indian festival for a given year."""
    calendar = {
        'diwali':    f'October/November {year} (lunar calendar)',
        'holi':      f'March {year} (lunar calendar)',
        'eid':       f'April/May {year} (moon-sighting dependent)',
        'christmas': f'December 25, {year}',
    }
    return calendar.get(festival.lower(), f'Date for {festival} not available.')

@tool
def get_state_capital(state: str) -> str:
    """Return the capital city of an Indian state."""
    capitals = {
        'maharashtra': 'Mumbai (executive) / Nagpur (winter session)',
        'tamil nadu':  'Chennai',
        'kerala':      'Thiruvananthapuram',
        'west bengal': 'Kolkata',
        'rajasthan':   'Jaipur',
    }
    return capitals.get(state.lower(), f'Capital for {state} not found.')

lc_llm_tools = lc_llm.bind_tools([get_festival_date, get_state_capital])
response = lc_llm_tools.invoke('When is Diwali in 2025 and what is the capital of Kerala?')

print('Tool calls requested by the model:')
for tc in response.tool_calls:
    print(f'  {tc["name"]}({tc["args"]})')


Tool calls requested by the model:
  get_festival_date({'festival': 'Diwali', 'year': 2025})


---
## Section 3 - Building Agents

> **Prerequisite:** Run Section 2 first so `lc_llm` is available.

> 💡 **Model tip - use `sarvam-105b` for production agents.** The 105B model scores significantly higher on agentic benchmarks: BrowseComp 49.5 vs 35.5 (30B) and Tau2 avg 68.3. It is optimized for tool use, long-horizon reasoning, and environment interaction. Switch by setting `MODEL = 'sarvam-105b'` in the setup cell - no other code changes needed.


### 3.1 Multi-Agent Handoffs

A lightweight triage agent classifies each query and routes it to a specialist: history, science, or culture.


In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def make_specialist(system_prompt: str):
    return (
        ChatPromptTemplate.from_messages([
            ('system', system_prompt),
            ('human', '{query}'),
        ])
        | lc_llm
        | StrOutputParser()
    )

history_agent  = make_specialist('You are a specialist in Indian history. Answer in 2-3 sentences.')
science_agent  = make_specialist('You are a specialist in Indian science and technology. Answer in 2-3 sentences.')
culture_agent  = make_specialist('You are a specialist in Indian arts and culture. Answer in 2-3 sentences.')

triage_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'Classify the query into exactly one word: history, science, or culture.'),
        ('human', '{query}'),
    ])
    | lc_llm
    | StrOutputParser()
)

def route(query: str) -> str:
    category = triage_chain.invoke({'query': query}).strip().lower()
    if 'science' in category:
        return science_agent.invoke({'query': query})
    elif 'culture' in category:
        return culture_agent.invoke({'query': query})
    return history_agent.invoke({'query': query})

for q in [
    'Tell me about the Chandrayaan-3 lunar mission.',
    'What are the origins of Bharatanatyam?',
    'What was the extent of the Maurya Empire?',
]:
    print(f'Q: {q}')
    print(f'A: {route(q)}\n')


Q: Tell me about the Chandrayaan-3 lunar mission.
A: The Chandrayaan-3 mission, conducted by the Indian Space Research Organisation (ISRO), successfully achieved the first-ever soft landing on the Moon's surface near the lunar south pole on August 23, 2023. The mission, consisting of the lander Vikram and the rover Pragyan, marked a historic milestone for India, making it the fourth country to achieve a lunar soft landing and the first to target the south pole.

Q: What are the origins of Bharatanatyam?
A: Bharatanatyam originates from the ancient treatise, the *Natya Shastra*, which codified the principles of Indian performing arts. It evolved over centuries as the temple dance tradition of the *devadasis* (temple dancers) in Tamil Nadu, known as *sadir* or *dasi attam*. This sacred art form was formally revived in the 20th century, shaping the classical dance form celebrated on the modern stage today.

Q: What was the extent of the Maurya Empire?
A: At its zenith under Ashoka, the Ma

### 3.2 Agents as Tools

Wrap specialist agents as `@tool` functions and hand them to an orchestrator. The orchestrator decides which expert to consult for each sub-question.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from langgraph.prebuilt import create_react_agent

@tool
def ask_history_expert(question: str) -> str:
    """Ask the Indian history specialist agent a question."""
    return history_agent.invoke({"query": question})

@tool
def ask_culture_expert(question: str) -> str:
    """Ask the Indian culture and arts specialist agent a question."""
    return culture_agent.invoke({"query": question})

orchestrator = create_react_agent(lc_llm, tools=[ask_history_expert, ask_culture_expert])

result = orchestrator.invoke({
    "messages": [{
        "role": "user",
        "content": (
            'Compare the historical significance of the Gupta Empire '
            'with its cultural contributions during the same era.'
        )
    }]
})
print(result["messages"][-1].content)


### 3.3 Agent with Input Guardrails

A guardrail LLM call classifies each query before it reaches the main agent. Off-topic queries are blocked with a clear rejection message.


In [25]:
guardrail_chain = (
    ChatPromptTemplate.from_messages([
        ('system',
         'You are a safety classifier. '
         'Decide if the query is related to India (culture, history, geography, language, science). '
         'Reply with only ALLOWED or BLOCKED.'),
        ('human', '{query}'),
    ])
    | lc_llm
    | StrOutputParser()
)

knowledge_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'You are a helpful guide specialising in India-related topics.'),
        ('human', '{query}'),
    ])
    | lc_llm
    | StrOutputParser()
)

def guarded_agent(query: str) -> str:
    verdict = guardrail_chain.invoke({'query': query}).strip().upper()
    if 'ALLOWED' in verdict:
        return knowledge_chain.invoke({'query': query})
    return '\u26d4 Blocked: This assistant only answers questions related to India.'

# Allowed query
print(guarded_agent('What is the significance of the Taj Mahal?'))
print()
# Blocked query
print(guarded_agent('Write a travel guide for Paris.'))


Of course. The Taj Mahal is one of the most significant monuments in the world, and its importance extends far beyond being just a beautiful tomb. It is a powerful symbol with deep historical, architectural, cultural, and spiritual meaning.

Here is a breakdown of its significance:

### 1. A Monument to Eternal Love
The most famous story behind the Taj Mahal is that of an emperor's undying love for his favourite wife, Mumtaz Mahal. After her death in 1631, the Mughal Emperor Shah Jahan commissioned the construction of an unparalleled mausoleum in her memory. While the exact details are debated by historians, the narrative of a monument built out of profound grief and love is central to its identity. This romantic story makes it an iconic symbol of love and devotion.

### 2. A Pinnacle of Mughal Architecture
The Taj Mahal is considered the zenith of Mughal architecture and a masterpiece of design. Its significance lies in its:
*   **Symmetry and Balance:** The entire complex is built on

### 3.4 Wiki-Grounded Search Agent

`wiki_grounding=True` is a Sarvam-specific parameter that grounds the model's response in Wikipedia-sourced facts. Ideal for factual knowledge-retrieval tasks where accuracy is critical.


In [27]:
def wiki_grounded_search(query: str) -> str:
    """Run a factual query with wiki grounding enabled."""
    response = client.chat.completions(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'You are a factual information assistant. Answer accurately and concisely.'},
            {'role': 'user',   'content': query},
        ],
        wiki_grounding=True,
        temperature=0.1,
        max_tokens=2000,  # increased for reasoning model
    )
    return response.choices[0].message.content

for q in [
    'What are the founding year and headquarters of the Reserve Bank of India?',
    'What is the total route length of the Indian Railways network?',
]:
    print(f'Query:  {q}')
    print(f'Answer: {wiki_grounded_search(q)}\n')


Query:  What are the founding year and headquarters of the Reserve Bank of India?
Answer: The Reserve Bank of India was established on **April 1, 1935**.

Its headquarters is in **Mumbai, Maharashtra**.

Query:  What is the total route length of the Indian Railways network?
Answer: The total route length of the Indian Railways network is **68,000 kilometers**.

This makes it one of the largest railway networks in the world, second only to China's.



### 3.5 LangGraph Agent

`create_react_agent` builds a full ReAct graph. Iterating over `result['messages']` shows the complete reasoning trace: user → tool calls → tool results → final answer.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from langgraph.prebuilt import create_react_agent

@tool
def search_landmark(landmark: str) -> str:
    """Return factual information about a famous Indian landmark."""
    db = {
        'taj mahal':        'Built in Agra by Mughal emperor Shah Jahan in 1632 as a mausoleum for Mumtaz Mahal.',
        'red fort':         'Built in Delhi by Shah Jahan in 1638; a UNESCO World Heritage Site.',
        'gateway of india': "Built in Mumbai in 1924 to commemorate King George V's visit in 1911.",
        'qutub minar':      'A 73-metre minaret in Delhi, begun in 1193 CE; UNESCO World Heritage Site.',
    }
    return db.get(landmark.lower(), f'No data found for "{landmark}".')

@tool
def convert_inr_to_usd(amount_inr: float) -> str:
    """Convert Indian Rupees (INR) to US Dollars at an approximate rate."""
    rate = 0.012  # ~83 INR per USD
    return f'₹{amount_inr:,.2f} ≈ ${amount_inr * rate:,.2f} USD (at ~₹83/USD)'

lg_agent = create_react_agent(lc_llm, tools=[search_landmark, convert_inr_to_usd])
result = lg_agent.invoke({
    "messages": [{
        "role": "user",
        'content': 'Tell me about the Gateway of India and convert ₹5000 to USD.'
    }]
})

for msg in result["messages"]:
    role    = getattr(msg, "type", "unknown").upper()
    content = msg.content or f'[tool_calls: {getattr(msg, "tool_calls", [])}]'
    print(f"[{role}] {str(content)[:300]}")
    print()


### 3.6 Native Tool Calling (Sarvam SDK)

Pass a JSON-schema tool definition directly to `client.chat.completions()`. The model returns structured `tool_calls`; your code executes the function and can feed the result back for a final response.


In [29]:
tools_schema = [
    {
        'type': 'function',
        'function': {
            'name': 'get_train_status',
            'description': 'Get the running status of an Indian Railways train by train number.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'train_number': {
                        'type': 'string',
                        'description': 'The 5-digit Indian Railways train number'
                    },
                    'date': {
                        'type': 'string',
                        'description': 'Travel date in YYYY-MM-DD format'
                    }
                },
                'required': ['train_number', 'date']
            }
        }
    }
]

response = client.chat.completions(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Check the status of train 12301 for 2025-04-15.'}],
    tools=tools_schema,
    tool_choice='auto',
)

tool_calls = response.choices[0].message.tool_calls
if tool_calls:
    tc = tool_calls[0]
    print(f'Tool invoked : {tc.function.name}')
    print(f'Arguments   : {tc.function.arguments}')
else:
    print('No tool called. Response:', response.choices[0].message.content)


Tool invoked : get_train_status
Arguments   : {"train_number": "12301", "date": "2025-04-15"}


---
## Section 4 - Advanced Applications


### 4.1 Structured Output via SDK + Pydantic

Request a JSON response using `response_format={"type": "json_object"}` and parse the output into a Pydantic model for type-safe access.


In [32]:
from pydantic import BaseModel, ValidationError
from typing import List
import re

class BookRecommendation(BaseModel):
    title: str
    author: str
    language: str
    genre: str
    brief_summary: str

prompt = (
    'Recommend one classic Indian novel. '
    'Return ONLY a valid JSON object with fields: title, author, language, genre, brief_summary.'
)

response = oa_client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'Always respond with valid JSON only.'},
        {'role': 'user',   'content': prompt},
    ],
    response_format={'type': 'json_object'},
    temperature=0.2,
    max_tokens=2000,
)

raw = response.choices[0].message.content or ''
# Strip markdown code fences if present
raw = re.sub(r'^```(?:json)?\s*', '', raw.strip())
raw = re.sub(r'\s*```$', '', raw)
try:
    book = BookRecommendation(**json.loads(raw))
    print(f'Title:         {book.title}')
    print(f'Author:        {book.author}')
    print(f'Language:      {book.language}')
    print(f'Genre:         {book.genre}')
    print(f'Brief summary: {book.brief_summary}')
except (json.JSONDecodeError, ValidationError) as e:
    print(f'Parsing error: {e}\nRaw response:\n{raw}')


Title:         The Guide
Author:        R.K. Narayan
Language:      English
Genre:         Literary Fiction
Brief summary: A spiritual guide becomes a tour guide for a tourist group in this novel about transformation and redemption. When Raju, a former tour guide, meets Rosie, a dancer, he begins a relationship that leads him to become a spiritual guide after his imprisonment. The novel explores themes of faith, identity, and the search for meaning in modern India.


### 4.2 Function Calling (Raw JSON Schema)

Define tools using raw JSON schema objects. The model returns `tool_calls` with the chosen function name and arguments as a JSON string; parse with `json.loads()` to obtain a typed dict.


In [33]:
tools_raw = [
    {
        'type': 'function',
        'function': {
            'name': 'calculate_gst',
            'description': 'Calculate Goods and Services Tax (GST) for a given base amount and rate in India.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'amount':   {'type': 'number', 'description': 'Base price in INR'},
                    'gst_rate': {'type': 'number', 'description': 'GST percentage, e.g. 5, 12, 18, or 28'},
                },
                'required': ['amount', 'gst_rate']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'format_postal_address',
            'description': 'Format an Indian postal address with a PIN code.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'street':   {'type': 'string'},
                    'city':     {'type': 'string'},
                    'state':    {'type': 'string'},
                    'pin_code': {'type': 'string'},
                },
                'required': ['street', 'city', 'state', 'pin_code']
            }
        }
    }
]

response = oa_client.chat.completions.create(
    model=MODEL,
    messages=[{
        'role': 'user',
        'content': 'Calculate 18% GST on \u20b92500, and format: 42 MG Road, Bengaluru, Karnataka, 560001.',
    }],
    tools=tools_raw,
    tool_choice='auto',
)

for tc in response.choices[0].message.tool_calls or []:
    args = json.loads(tc.function.arguments)
    print(f'Function : {tc.function.name}')
    print(f'Arguments: {json.dumps(args, indent=2)}')
    print()


Function : calculate_gst
Arguments: {
  "amount": 2500,
  "gst_rate": 18
}



### 4.3 RAG Pipeline

A Retrieval-Augmented Generation pipeline: an in-memory retriever fetches context documents, a `ChatPromptTemplate` injects them alongside the question, and the LLM answers strictly from context.


In [34]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

KNOWLEDGE_BASE = [
    'The Maurya Empire (322-185 BCE) was the first pan-Indian empire, founded by Chandragupta Maurya with counsel from Chanakya.',
    'Emperor Ashoka expanded the empire across most of the subcontinent. After the Kalinga War he converted to Buddhism.',
    "The Gupta Empire (320-550 CE) is called India's Golden Age. Literature, science, and mathematics flourished.",
    'Aryabhata (476-550 CE) estimated pi to 4 decimal places and proposed Earth rotates on its own axis.',
    'Kalidasa, the Sanskrit poet and dramatist, wrote Shakuntala and Meghaduta during the Gupta period.',
]

def retrieve(query: str) -> str:
    relevant = [doc for doc in KNOWLEDGE_BASE
                if any(w.lower() in doc.lower() for w in query.split() if len(w) > 4)]
    return '\n\n'.join(relevant) if relevant else '\n\n'.join(KNOWLEDGE_BASE[:2])

rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer using ONLY the provided context. If it is insufficient, say "Not found in context".'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}'),
])

rag_chain = (
    {'context': RunnableLambda(retrieve), 'question': RunnablePassthrough()}
    | rag_prompt
    | lc_llm
    | StrOutputParser()
)

print(rag_chain.invoke('What did Aryabhata discover?'))


Aryabhata estimated pi to 4 decimal places and proposed that Earth rotates on its own axis.


### 4.4 Content Generation Pipeline

A two-step sequential chain: Step 1 generates blog title ideas; Step 2 writes an introduction for the first idea. The output of Step 1 is piped as input to Step 2.


In [35]:
idea_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a creative content strategist specialising in Indian culture.'),
    ('human', 'Generate 3 catchy blog post titles about {topic}. Return only a numbered list.'),
])

write_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an engaging travel and culture writer.'),
    ('human', 'Write a 3-paragraph blog introduction for the FIRST title in this list:\n\n{titles}'),
])

idea_chain = idea_prompt | lc_llm | StrOutputParser()

full_chain = (
    idea_chain
    | (lambda titles: {'titles': titles})
    | write_prompt
    | lc_llm
    | StrOutputParser()
)

print(full_chain.invoke({'topic': 'ancient Indian temples'}))


Forget what you know about temples as mere stone and mortar. As you stand before the ancient temples of India, you are not just looking at a structure; you are standing at the threshold of a deep, resonant silence. It’s a silence that, if you listen closely, is not empty at all. It’s filled with the faint, persistent “whispers” of a thousand years of devotion, artistry, and history. These are the stone whispers, the secrets encoded in the weathered carvings and the silent, patient gaze of the sculptures, waiting for a modern pilgrim to decipher their ancient language.

These are not secrets of lost treasure, but of the human soul. Unlocking them means tracing the path of the ancient sages who envisioned these cosmic microcosms, the kings who commissioned them as acts of faith, and the countless artisans who poured their genius into every intricate detail. Each whisper tells a story of a forgotten festival, a celestial alignment, a philosophical debate, and a prayer offered up in stone.

### 4.5 Async Batch Processing

`AsyncOpenAI` combined with `asyncio.gather()` fires all prompts concurrently for near-linear throughput scaling. Define `async def main()` and call it with `await main()` inside a Jupyter cell.


In [37]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI(
    api_key=os.environ['SARVAM_API_KEY'],
    base_url=SARVAM_BASE_URL,
    timeout=300,
)

prompts = [
    'In one sentence, what is yoga?',
    'In one sentence, what is Ayurveda?',
    'In one sentence, what is Vedanta philosophy?',
    'In one sentence, what is the significance of the River Ganga?',
]

async def main():
    tasks = [
        async_client.chat.completions.create(
            model=MODEL,
            messages=[{'role': 'user', 'content': p}],
            max_tokens=2000,  # increased for reasoning model
        )
        for p in prompts
    ]
    results = await asyncio.gather(*tasks)
    for prompt, result in zip(prompts, results):
        print(f'Q: {prompt}')
        print(f'A: {result.choices[0].message.content}')
        print()

await main()


Q: In one sentence, what is yoga?
A: Yoga is a practice that unites the mind, body, and spirit through physical postures, breathing techniques, and meditation.

Q: In one sentence, what is Ayurveda?
A: Ayurveda is an ancient system of holistic medicine from India that uses diet,

Q: In one sentence, what is Vedanta philosophy?
A: Vedanta is the Hindu philosophical tradition that teaches the individual soul (Atman) is identical to the ultimate reality (Brahman), with the ultimate goal of achieving liberation (Moksha) from the cycle of rebirth.

Q: In one sentence, what is the significance of the River Ganga?
A: The River Ganga is the sacred and spiritual heart of India, a life-giving force that has sustained civilizations and shaped the nation's cultural and religious identity for millennia.



### 4.6 JSON Mode

Set `response_format={"type": "json_object"}` to guarantee the model returns a parseable JSON string. Always include a system instruction to reinforce the format.


In [40]:
import re

response = oa_client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'Always respond with valid JSON only.'},
        {'role': 'user',   'content': (
            'List 3 Indian UNESCO World Heritage Sites as a JSON array. '
            'Each object must have: name, location, year_inscribed.'
        )},
    ],
    response_format={'type': 'json_object'},
    max_tokens=2000,  # increased for reasoning model
)

raw_json = response.choices[0].message.content or ''
# Strip markdown code fences if present
raw_json = re.sub(r'^```(?:json)?\s*', '', raw_json.strip())
raw_json = re.sub(r'\s*```$', '', raw_json)
try:
    data = json.loads(raw_json)
    print(json.dumps(data, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f'JSON parse error: {e}\nRaw output:\n{raw_json}')


[
  {
    "name": "Taj Mahal",
    "location": "Agra, Uttar Pradesh",
    "year_inscribed": "1983"
  },
  {
    "name": "Qutub Minar",
    "location": "Delhi",
    "year_inscribed": "1997"
  },
  {
    "name": "Ajanta and Ellora Caves",
    "location": "Aurangabad, Maharashtra",
    "year_inscribed": "1983"
  }
]


### 4.7 Hybrid Thinking Mode (`reasoning_effort`) ★ Sarvam-unique

Set `reasoning_effort` to `'low'`, `'medium'`, or `'high'` to activate the model's internal chain-of-thought. Thinking steps are returned in `choices[0].message.reasoning_content` alongside the final answer in `content`. Higher effort improves accuracy on math, coding, and multi-step logic at the cost of higher latency.

> Supported on both **`sarvam-30b-16k`** and **`sarvam-105b`**. With 105B at `high` reasoning effort, the model reaches 98.6 on Math500 and 96.7 on AIME 25 (with tools) - among the strongest scores of any model of its class.


In [42]:
response_thinking = client.chat.completions(
    model=MODEL,
    messages=[{
        'role': 'user',
        'content': (
            'A farmer has rice fields of 12.5 hectares. '
            'He harvests 3.2 tonnes per hectare. '
            'He sells 60% of his harvest at ₹2200 per quintal and stores the rest. '
            'How many quintals does he store and what is his total revenue from sales?'
        )
    }],
    reasoning_effort='medium',
    temperature=0.2,
    max_tokens=4096,  # increased to allow reasoning + final answer
)

reasoning = response_thinking.choices[0].message.reasoning_content
answer    = response_thinking.choices[0].message.content

if reasoning:
    preview = reasoning[:600] + ('...' if len(reasoning) > 600 else '')
    print('=== Thinking Steps (preview) ===')
    print(preview)
    print()

print('=== Final Answer ===')
print(answer)


=== Thinking Steps (preview) ===
1.  **Deconstruct the User's Request:**
    *   **Goal:** Calculate two things:
        1.  How many quintals of rice does the farmer store?
        2.  What is his total revenue from sales?
    *   **Given Information:**
        *   Area of rice fields: 12.5 hectares
        *   Harvest yield: 3.2 tonnes per hectare
        *   Sales percentage: 60% of the total harvest
        *   Sales price: ₹2200 per quintal
    *   **Implicit Information/Units:**
        *   The final answer for storage needs to be in "quintals".
        *   The revenue needs to be in a monetary unit (₹).
        *   The...

=== Final Answer ===
Based on the information provided, here is the calculation broken down step-by-step.

### Final Answer

*   **Quintals Stored:** 160 quintals
*   **Total Revenue from Sales:** ₹52,800

---

### Step-by-Step Calculation

**Step 1: Calculate the total harvest in tonnes.**

*   Area of fields: 12.5 hectares
*   Yield per hectare: 3.2 tonnes
*

### 4.8 Wiki Grounding ★ Sarvam-unique

`wiki_grounding=True` instructs the model to cross-reference its answer against Wikipedia. Compare outputs with and without grounding on the same factual query.

> Supported on both **`sarvam-30b-16k`** and **`sarvam-105b`**. With 105B, wiki grounding combines with the model's deeper knowledge (12T token pre-training) for highly accurate factual answers.


In [44]:
QUERY = 'What are the key facts about the Indian Space Research Organisation (ISRO)?'

def ask(wiki: bool) -> str:
    response = client.chat.completions(
        model=MODEL,
        messages=[{'role': 'user', 'content': QUERY}],
        wiki_grounding=wiki,
        temperature=0.1,
        max_tokens=2000,  # increased for reasoning model
    )
    return response.choices[0].message.content

print('=== With Wiki Grounding ===')
print(ask(wiki=True))
print()
print('=== Without Wiki Grounding ===')
print(ask(wiki=False))


=== With Wiki Grounding ===
Of course! Here are the key facts about the Indian Space Research Organisation (ISRO), structured for clarity.

### **1. Basic Information**

*   **Full Name:** Indian Space Research Organisation (ISRO)
*   **Acronym:** ISRO
*   **Headquarters:** Bengaluru, Karnataka
*   **Founded:** August 15, 1969
*   **Founder:** Dr. Vikram Sarabhai
*   **Motto:** "Satyam, Shivam, Sundaram" (Truth, Goodness, Beauty)

### **2. History and Vision**

*   **Predecessor:** ISRO was formed by taking over the functions of the **Indian National Committee for Space Research (INCOSPAR)**, which was established in 1962.
*   **Founder's Vision:** Dr. Vikram Sarabhai, widely regarded as the father of the Indian space program, envisioned using space technology for the nation's development. He believed that space research could be a powerful tool for communication, resource management, and improving the quality of life for common people.
*   **Initial Focus:** The early years were focus

### 4.9 Indian Language Support ★ Sarvam-unique

Sarvam models were post-trained on the 10 most-spoken Indian languages. They accept native script, transliterated (romanized), and code-mixed (e.g., Hinglish) inputs and respond in the same register.

> **`sarvam-30b`** wins 89% of pairwise comparisons on Indian language benchmarks.  
> **`sarvam-105b`** wins 90% - the highest Indian language win rate of any compared model (open or closed-source).


In [46]:
examples = [
    # Native Devanagari script (Hindi)
    {'lang': 'Hindi - Devanagari',    'content': 'भारत की राजधानी क्या है?'},
    # Romanized Hindi
    {'lang': 'Hindi - Romanized',     'content': 'Bharat ki rajdhani kya hai?'},
    # Hinglish (code-mixed)
    {'lang': 'Hinglish - Code-mixed', 'content': 'India ka capital kya hai aur wahan ka weather kaisa hota hai?'},
    # Tamil (native script)
    {'lang': 'Tamil - Native',        'content': 'இந்தியாவின் தலைநகரம் எது?'},
    # Bengali (native script)
    {'lang': 'Bengali - Native',      'content': 'ভারতের রাজধানী কোথায়?'},
]

for ex in examples:
    response = client.chat.completions(
        model=MODEL,
        messages=[{'role': 'user', 'content': ex['content']}],
        max_tokens=2000,  # increased for reasoning model
        temperature=0.3,
    )
    print(f"[{ex['lang']}]")
    print(f"Input : {ex['content']}")
    print(f"Output: {response.choices[0].message.content}")
    print()


[Hindi - Devanagari]
Input : भारत की राजधानी क्या है?
Output: भारत की राजधानी **नई दिल्ली** है।

यद्यपि नई दिल्ली सरकार का केंद्र है, फिर भी अक्सर लोग मुंबई को भारत का सबसे बड़ा शहर और वित्तीय राजधानी मान लेते हैं। नई दिल्ली का आधिकारिक नाम 'राष्ट्रीय राजधानी क्षेत्र दिल्ली' (एन सी टी दिल्ली) है।

[Hindi - Romanized]
Input : Bharat ki rajdhani kya hai?
Output: Bharat ki rajdhani New Delhi hai. Ye ek kendriya shasan kshetra hai jo Bharat sarkar ka mukhya kendra hai.

[Hinglish - Code-mixed]
Input : India ka capital kya hai aur wahan ka weather kaisa hota hai?
Output: India ka capital **New Delhi** hai.

Yeh India ka official capital hai, jahan Parliament, Supreme Court, aur President ka residence (Rashtrapati Bhavan) located hai. Iske alawa, yeh National Capital Region (NCR) ka hissa hai, jo ek bahut bada aur bustling metropolitan area hai.

New Delhi mein saal bhar alag-alag mausam hote hain, jo chaar main seasons mein baante hote hain:

### 1. Summer (Garmi) - March se June
Yeh season

---
## ★ Quick-Reference Footer

| Concept | Value |
|---|---|
| **Model ID (standard)** | `sarvam-30b` |
| **Model ID (cost-efficient)** | `sarvam-30b-16k` |
| **Model ID (flagship)** | `sarvam-105b` |
| **Context window** | 64K tokens (`sarvam-30b`) / 16K tokens (`sarvam-30b-16k`) |
| **API base URL** | `https://api.sarvam.ai/v1` |
| **Chat completions endpoint** | `POST /v1/chat/completions` |
| **API key env variable** | `SARVAM_API_KEY` |
| **Native SDK package** | `sarvamai` |
| **Native client init** | `SarvamAI(api_subscription_key=...)` |
| **Native chat call** | `client.chat.completions(model=MODEL, messages=[...])` |
| **OpenAI-compat client** | `OpenAI(api_key=..., base_url=SARVAM_BASE_URL)` |
| **OpenAI-compat chat call** | `oa_client.chat.completions.create(model=MODEL, messages=[...])` |
| **Streaming** | `stream=True` |
| **Thinking mode** | `reasoning_effort='low'/'medium'/'high'` |
| **Thinking content field** | `choices[0].message.reasoning_content` |
| **Wiki grounding** | `wiki_grounding=True` |
| **Tool calling** | `tools=[...]`, `tool_choice='auto'` |
| **Tool result field** | `choices[0].message.tool_calls[i].function.{name,arguments}` |
| **JSON mode** | `response_format={'type': 'json_object'}` |
| **Temperature range** | 0–2 (default 0.2) |
| **Top-p range** | 0–1 (default 1) |
| **Reasoning effort options** | `low`, `medium`, `high` |
| **Supported Indian languages** | Hindi, Bengali, Tamil, Telugu, Marathi, Gujarati, Kannada, Malayalam, Odia, Punjabi |
| **Input formats** | Native script · Romanized · Code-mixed |
| **LangChain wrapper** | `ChatOpenAI(openai_api_base=SARVAM_BASE_URL, ...)` |
| **License** | Apache 2.0 |

### Sarvam-105B at a Glance

| Concept | Value |
|---|---|
| **Model ID** | `sarvam-105b` |
| **Architecture** | MoE + Multi-head Latent Attention (MLA), 128 sparse experts |
| **Total parameters** | 105B+ |
| **Pre-training data** | 12T tokens (code, math, multilingual, web) |
| **Math500** | 98.6 |
| **AIME 25** | 88.3 (96.7 with tools) |
| **BrowseComp** | 49.5 (vs 35.5 for 30B) |
| **Indian Language Win Rate** | 90% |
| **Best for** | Complex reasoning, agentic workflows, highest-quality outputs |
| **Inference** | Server-centric (H100) |
| **API compatibility** | Fully drop-in with all code in this cookbook |
| **License** | Apache 2.0 |

---

*Generated from the [Sarvam-30B](https://docs.sarvam.ai/api-reference-docs/getting-started/models/sarvam-30b) and [Sarvam-105B](https://docs.sarvam.ai/api-reference-docs/getting-started/models/sarvam-105b) documentation - March 9, 2026.*
